# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [18]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [19]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [21]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [22]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [23]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [24]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [25]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [26]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle, focus on stretching and strengthening, and can help alleviate lower back discomfort. However, please consult with a healthcare professional before starting any new exercise routine, especially if you have existing back issues.'

In [27]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It supports physical repair by enabling the body to heal tissues and regenerate cells, as well as consolidating memories and learning. Adequate sleep (7-9 hours for adults) is essential for mental well-being, cognitive functions, and emotional balance. Additionally, sleep helps regulate hormones that influence growth and appetite, strengthens the immune system, and contributes to emotional resilience. Poor sleep or sleep disorders like insomnia can negatively impact these processes, increasing the risk of health issues such as weakened immunity, mental health problems, and chronic conditions. Therefore, prioritizing good sleep hygiene and creating a conducive sleep environment are important strategies for promoting overall health.'

In [28]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying a cold or warm compress to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils such as peppermint or lavender\n- Taking small amounts of caffeine (with caution)\n- Practicing deep breathing exercises\n- Engaging in progressive muscle relaxation\n- Going for a short walk, preferably in nature\n- Listening to calming music\n\nThese techniques can help alleviate stress and headaches naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [29]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [30]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [31]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, then flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [32]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. Adequate sleep, typically 7-9 hours for adults, supports essential bodily processes such as tissue repair, brain function, and immune system strength. During sleep, the body cycles through stages including deep sleep (Stage 3), where physical repair and regeneration occur, and REM sleep, which is vital for memory and learning. Maintaining a consistent sleep schedule and creating an optimal sleep environment—such as keeping the room dark, quiet, and cool—can improve sleep quality. Poor sleep or conditions like insomnia can negatively impact overall health, affecting mental clarity, immune function, and physical well-being.'

In [33]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as progressive muscle relaxation, meditation, and deep breathing exercises. Herbal teas like chamomile or valerian root may also help promote relaxation. Additionally, staying well-hydrated and ensuring adequate sleep can reduce headache triggers related to stress. If considering supplements like magnesium, it’s best to consult with a healthcare provider first.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

An example where BM25 would perform better is: “Where in the document does it mention 10–15 reps for the Cat-Cow stretch?”

I think BM25 is likely stronger here because the query includes exact phrases and specific numbers that probably appear word-for-word in the text. Since BM25 relies on keyword matching, it does especially well when the wording in the question closely matches the wording in the document.

Embeddings are better at understanding meaning and handling paraphrased questions, but for very precise or exact-term searches, BM25 can often return more accurate results.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [34]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [35]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [37]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Based on the provided information, some exercises that can help with lower back pain include:\n\n1. **Cat-Cow Stretch**: Start on your hands and knees. Alternate between arching your back up (cat position) and letting it sag down (cow position). Aim for 10-15 repetitions.\n\n2. **Bird Dog**: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. **Pelvic Tilts**: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nAlways consult with a healthcare professional before starting new exercises, especially if you have chronic or severe back pain.'

In [38]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—also helps maintain a healthy immune system, emotional stability, and proper brain function. Poor or insufficient sleep can lead to health issues such as weakened immunity, impaired memory, increased risk of chronic conditions, and mental health problems.'

In [39]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music. For headaches specifically, staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, gentle massage of the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule can be effective.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [40]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [41]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [43]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle and targeted at strengthening and stretching muscles to alleviate lower back dis

In [44]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It supports physical recovery by allowing the body to repair tissues and regenerate cells. Sleep also consolidates memories and enhances cognitive functions. Additionally, during sleep, the body releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night, contributes to mental well-being, strengthens the immune system, and helps manage stress. Poor sleep or sleep disturbances can lead to a range of health issues, including fatigue, stress, immune dysfunction, and increased risk of chronic conditions. Therefore, maintaining good sleep hygiene and a consistent sleep routine is essential for overall health and well-being.'

In [45]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhale for 4 counts, hold for 4, exhale for 4)\n- Progressive muscle relaxation by tensing and releasing muscle groups\n- Grounding techniques, such as naming things you see, hear, feel, smell, and taste\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule and practicing calming bedtime routines\n\nThese strategies can help alleviate stress and headaches naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Creating multiple versions of the same user query can improve recall because each variation may connect with different parts of the documents. Sometimes the original question uses wording that doesn’t exactly match how the information appears in the text. By rephrasing the query in different ways, we increase the likelihood of retrieving relevant content.

In short, multiple reformulations broaden the search coverage and reduce the chance of missing useful information just because of wording differences.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [46]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [47]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [48]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [49]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [50]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [51]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each side for 5 seconds and do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and lift shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nAlways ensure to perform these exercises gently and consult wi

In [52]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—ensures these processes occur effectively, promoting optimal health and functioning. Poor sleep or insufficient sleep can lead to fatigue, impaired immune function, and increased risk of chronic conditions. Therefore, maintaining good sleep habits and quality sleep is essential for overall health.'

In [53]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in mindfulness or meditation, doing light stretching or yoga, taking a warm bath, and using relaxation techniques like progressive muscle relaxation. For headaches specifically, remedies such as applying a cold or warm compress to the head or neck, resting in a dark and quiet room, gentle massage of the temples and neck, and using essential oils like peppermint or lavender can be helpful. Additionally, staying well-hydrated and maintaining a regular sleep schedule are important strategies to manage both stress and headaches naturally.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [54]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [55]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [56]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help alleviate lower back pain include:\n\n1. Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n2. Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n3. Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\n4. Partial Crunches: Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n5. Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and aimed at stretching and strengthenin

In [57]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It is essential for physical repair, mental well-being, and cognitive functions such as memory and learning. During sleep, the body repairs tissues, regulates hormones related to growth and appetite, and consolidates memories. Adequate sleep, typically 7-9 hours per night for adults, supports a strong immune system, improves mood, reduces stress, and enhances daily functioning. Poor sleep or sleep disturbances like insomnia can negatively impact health, increasing the risk of chronic conditions, affecting mental health, and impairing immune function. Maintaining good sleep hygiene, creating an optimal sleep environment, and addressing sleep issues are important for overall wellness.'

In [58]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises\n- Progressive muscle relaxation\n- Grounding techniques (naming things you see, hear, feel, smell, taste)\n- Taking short walks, especially in nature\n- Listening to calming music\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing mindfulness and meditation\n\nThese methods can help reduce stress and alleviate headache symptoms naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [59]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [60]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [61]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [62]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [63]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [64]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises are recommended for lower back pain relief and can help prevent future episodes.'

In [65]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. It is during sleep that the body works to repair tissues, regulate hormones, and consolidate memories. Adequate sleep—typically 7 to 9 hours per night for adults—is essential for physical health, mental well-being, and cognitive function. Poor sleep or sleep disturbances can lead to issues such as fatigue, headaches, difficulty concentrating, weakened immune function, and increased risk of chronic diseases. Therefore, quality sleep is foundational to overall health and wellness.'

In [66]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking plenty of water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Practicing relaxation techniques like deep breathing, progressive muscle relaxation, and grounding exercises\n- Engaging in physical activity like a short walk in nature\n- Maintaining a consistent sleep schedule and creating a relaxing bedtime routine\n\nThese approaches can help reduce stress and alleviate headache symptoms naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

If the sentences are very short and repetitive, semantic chunking may group too many of them together because their embeddings will look very similar. Since the algorithm relies on semantic differences to decide where to split, it might not detect clear boundaries and end up creating overly large chunks.

To fix this, I would adjust the similarity threshold to be more strict so that even small differences can trigger a split. I could also add a rule for maximum chunk length to make sure chunks don’t become too large, even if the sentences are semantically close.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE

# Imports + Metrics

import time
import numpy as np
import pandas as pd
from tqdm import tqdm

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.metrics import context_precision, context_recall  # legacy but works

from ragas.testset import TestsetGenerator
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI

METRICS = [context_precision, context_recall]

/tmp/ipykernel_2209/1863836979.py:13: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall  # legacy but works
/tmp/ipykernel_2209/1863836979.py:13: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall  # legacy but works


I import all required libraries for data handling, timing, evaluation, and RAGAS. 
I also define the evaluation metrics (context_precision and context_recall) that we will use to compare retrievers.

In [ ]:
# Generate a RAGAS testset

generator_llm = ChatOpenAI(model="gpt-4.1-nano")
critic_llm = ChatOpenAI(model="gpt-4.1-nano")

ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)  # assumes `embeddings` exists
generator = TestsetGenerator.from_langchain(generator_llm, critic_llm, ragas_embeddings)

TESTSET_SIZE = 20
testset = generator.generate_with_langchain_docs(
    raw_docs,  # assumes `raw_docs` exists (LangChain docs)
    testset_size=TESTSET_SIZE,
    transforms_embedding_model=ragas_embeddings
)

testset

/tmp/ipykernel_2209/1726409169.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)  # assumes `embeddings` exists


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/22 [00:00<?, ?it/s]

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input="What are the key reasons to visit Shanghai for health and wellness activities, and how can I incorporate exercise routines inspired by the city's lifestyle to improve my overall well-being?", retrieved_contexts=None, reference_contexts=['PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: Understanding Exercise Basics\n\nExercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.\n\nThe four main types of exercise are aerobic (cardio), strength training, flexibility, and balance exercises. A well-rounded fitness routine includes all four types. Adults should aim for at least 150 minutes of moderate-intensity aerobic activity per week, along with muscle-strengthening activities on 2 or more days per week.\n\nChapter 2: Exercise

I use RAGAS with a generator and critic LLM to automatically create a synthetic evaluation dataset from my source documents. This gives me realistic questions and ground-truth answers for benchmarking.

In [ ]:
# Convert testset -> golden CSV (question + ground_truth)

gold_df = (
    testset.to_pandas()
    .rename(columns={"user_input": "question", "reference": "ground_truth"})
    [["question", "ground_truth"]]
    .dropna()
    .reset_index(drop=True)
)

out_path = "data/golden_dataset.csv"
gold_df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(gold_df)}")

gold_df.head(10)

Saved: data/golden_dataset.csv
Rows: 22


,question,ground_truth
0,What are the key reasons to visit Shanghai for...,The provided context does not include specific...
1,How Shanghai help with health stuff?,The context provided does not mention Shanghai...
2,so like what is exercise and movement and why ...,Exercise and movement are important for your h...
3,What role do carbohydrates play in a balanced ...,Carbohydrates are the primary energy source in...
4,Whaat is the importnace of NUTRITION AND DIET ...,A balanced diet provides your body with the nu...
5,What r macronutrients?,Macronutrients are the key components of a bal...
6,What information is covered in Chapter 16 of t...,Chapter 16 discusses managing headaches natura...
7,What is Chapter 13 about?,Chapter 13 discusses the science of habit form...
8,What is PART 5 about in building healthy habits?,PART 5 discusses building healthy habits by ex...
9,What does PART 4 cover in the context of stres...,PART 4 focuses on stress management and mental...


I convert the RAGAS testset into a clean DataFrame with only question and ground_truth. Then I save it locally as a CSV file to ensure reproducibility and avoid regenerating it later.

In [ ]:
# Helpers: mean + build eval df for a retriever

def mean_value(x):
    """RAGAS can return list (per question) or scalar (aggregate)."""
    if isinstance(x, (list, tuple)):
        vals = [v for v in x if v is not None]
        return float(np.mean(vals)) if vals else float("nan")
    return float(x)

def build_eval_df(retriever, gold_df, n_questions=10):
    """
    Returns:
      eval_df columns: user_input, retrieved_contexts, reference
      timing dict: retrieval_total_sec, retrieval_sec_per_q, n_questions
    """
    subset = gold_df.head(n_questions).copy()
    questions = subset["question"].tolist()

    rows = []
    t0 = time.perf_counter()

    for q in tqdm(questions, desc="Retrieving"):
        docs = retriever.get_relevant_documents(q)
        rows.append({
            "user_input": q,
            "retrieved_contexts": [d.page_content for d in docs],
        })

    retrieval_total = time.perf_counter() - t0
    retrieval_per_q = retrieval_total / max(len(questions), 1)

    retrieval_df = pd.DataFrame(rows)

    # attach references
    ref_df = subset.rename(columns={"question": "user_input", "ground_truth": "reference"})[
        ["user_input", "reference"]
    ]

    eval_df = retrieval_df.merge(ref_df, on="user_input", how="left").dropna(subset=["reference"]).reset_index(drop=True)

    timing = {
        "retrieval_total_sec": retrieval_total,
        "retrieval_sec_per_q": retrieval_per_q,
        "n_questions": len(eval_df),
    }
    return eval_df, timing

For a selected subset of questions, I retrieve relevant document chunks using a specific retriever and then combine the retrieved contexts with the correct reference answers and measure retrieval latency.

In [ ]:
# Run RAGAS evaluation on eval_df

def eval_with_ragas(eval_df, ragas_embeddings):
    """
    Returns:
      scores (dict-like) + timing dict
    """
    dataset = EvaluationDataset.from_list(eval_df.to_dict(orient="records"))

    t0 = time.perf_counter()
    scores = evaluate(dataset, metrics=METRICS, embeddings=ragas_embeddings)
    total = time.perf_counter() - t0

    timing = {
        "ragas_total_sec": total,
        "ragas_sec_per_q": total / max(len(eval_df), 1),
    }
    return scores, timing

I convert the retrieval results into a RAGAS EvaluationDataset and compute retrieval metrics (precision and recall). I also measure how long the evaluation process takes per question.

In [84]:
# Benchmark multiple retrievers + summary table

def benchmark_retriever(name, retriever, gold_df, ragas_embeddings, n_questions=10):
    eval_df, t_retr = build_eval_df(retriever, gold_df, n_questions=n_questions)
    scores, t_eval = eval_with_ragas(eval_df, ragas_embeddings)

    total_per_q = t_retr["retrieval_sec_per_q"] + t_eval["ragas_sec_per_q"]

    return {
        "retriever": name,
        "n_questions": t_retr["n_questions"],
        "context_precision": mean_value(scores["context_precision"]),
        "context_recall": mean_value(scores["context_recall"]),
        "retrieval_sec_per_q": round(t_retr["retrieval_sec_per_q"], 2),
        "ragas_sec_per_q": round(t_eval["ragas_sec_per_q"], 2),
        "total_sec_per_q": round(total_per_q, 2),
    }

N = 10  # same questions for fairness

strategies = [
    ("naive", naive_retriever),
    ("bm25", bm25_retriever),
    ("parent", parent_document_retriever),
    ("multi_query", multi_query_retriever),
    ("compression", compression_retriever),
    ("ensemble", ensemble_retriever),
]

results = [benchmark_retriever(name, r, gold_df, ragas_embeddings, n_questions=N) for name, r in strategies]

summary = (
    pd.DataFrame(results)
    .sort_values(by="context_recall", ascending=False)
    .reset_index(drop=True)
)

summary

Retrieving:   0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_2209/2267827596.py:24: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(q)
Retrieving: 100%|██████████| 10/10 [00:07<00:00,  1.43it/s]


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Retrieving: 100%|██████████| 10/10 [00:00<00:00, 3128.21it/s]


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Retrieving: 100%|██████████| 10/10 [00:14<00:00,  1.42s/it]


...

Retrieving: 100%|██████████| 10/10 [00:10<00:00,  1.07s/it]


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Retrieving: 100%|██████████| 10/10 [01:02<00:00,  6.29s/it]


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

,retriever,n_questions,context_precision,context_recall,retrieval_sec_per_q,ragas_sec_per_q,total_sec_per_q
0,parent,10,0.713889,1.000000,1.42,12.18,13.60
1,ensemble,10,0.538294,0.966667,6.29,50.69,56.97
2,naive,10,0.746111,0.890476,0.70,30.72,31.42
3,multi_query,10,0.652629,0.784524,2.16,37.87,40.03
4,compression,10,0.783333,0.701190,1.07,13.73,14.80
5,bm25,10,0.350000,0.409524,0.00,15.03,15.03


I run the same evaluation process for multiple retriever strategies and collect their results. Finally, I created a summary table (sorted by recall) to compare quality and latency across all methods.

# Conclusion

In this activity, I compared several retriever strategies using a synthetic golden dataset and RAGAS metrics (context_precision and context_recall), along with latency per query.

The Parent Document Retriever gave the best overall balance. It achieved high precision and strong recall, meaning it retrieved mostly relevant content while still covering most of the correct information. Its latency was also reasonable compared to more complex methods.

The Ensemble retriever had the highest recall, but it introduced more noise and was much slower. Other methods like multi-query and compression improved recall compared to naive retrieval, but they increased latency because they rely on extra LLM steps.

Overall, the Parent retriever provided the best trade-off between quality and speed for this dataset.